# UAV Object Detection · Detectron2 lab
## 04 · Anchor boxes & regression deltas
Supports the **Anchor Box Lab**. Detectors don't regress boxes from nothing — they place reference **anchors** (scales × aspect ratios) at every feature location and predict small **offset deltas** (tx, ty, tw, th) toward the ground truth.

In [ ]:
%%writefile drone_synth.py
"""
drone_synth.py — contextualized synthetic drone (nadir/aerial) imagery.

Generates top-down scenes (parking lots and crop fields) populated with
cars, people and trees, plus COCO-format annotations (bbox + polygon
segmentation). Pure numpy + Pillow, so it runs with no GPU and no
Detectron2 install. The visual language mirrors the HTML study
playgrounds: magenta/blue accents, nadir vehicles, small people from
altitude, cluttered backgrounds.

Categories (COCO ids are 1-indexed):
    1 = car     2 = person     3 = tree

Key entry points:
    make_scene(seed, kind)          -> (PIL.Image, [ann, ...])
    build_coco(n, out_dir, split)   -> path to COCO json (+ images on disk)
    THING_CLASSES                   -> ["car", "person", "tree"]
"""
import json, math, os, random
import numpy as np
from PIL import Image, ImageDraw

THING_CLASSES = ["car", "person", "tree"]
W_DEF, H_DEF = 512, 512

# ---- palette (kept close to the HTML playgrounds) -------------------------
ASPHALT = (157, 162, 148)
FIELD   = (170, 182, 132)
CROP    = (139, 155, 113)
DIRT    = (176, 152, 122)
LANE    = (232, 228, 210)
CAUTION = (200, 150, 40)
CAR_COLORS = [(122, 48, 48), (47, 90, 138), (90, 106, 47), (90, 63, 110), (60, 60, 66)]


def _poly_from_ellipse(cx, cy, rx, ry, n=16):
    return [(cx + rx * math.cos(2 * math.pi * i / n),
             cy + ry * math.sin(2 * math.pi * i / n)) for i in range(n)]


def _rect_poly(x, y, w, h):
    return [(x, y), (x + w, y), (x + w, y + h), (x, y + h)]


def _draw_car(dr, box, color):
    x, y, w, h = box
    pad = min(w, h) * 0.06
    dr.rounded_rectangle([x + pad, y + pad, x + w - pad, y + h - pad],
                         radius=min(w, h) * 0.16, fill=color, outline=(14, 18, 22))
    vertical = h >= w
    if vertical:
        dr.rounded_rectangle([x + w * 0.22, y + h * 0.30, x + w * 0.78, y + h * 0.64],
                             radius=3, fill=(143, 160, 173))
        dr.rounded_rectangle([x + w * 0.26, y + h * 0.12, x + w * 0.74, y + h * 0.24],
                             radius=2, fill=(199, 210, 218))
    else:
        dr.rounded_rectangle([x + w * 0.30, y + h * 0.22, x + w * 0.64, y + h * 0.78],
                             radius=3, fill=(143, 160, 173))
        dr.rounded_rectangle([x + w * 0.12, y + h * 0.26, x + w * 0.24, y + h * 0.74],
                             radius=2, fill=(199, 210, 218))
    return _rect_poly(x + pad, y + pad, w - 2 * pad, h - 2 * pad)


def _draw_person(dr, box):
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    r = min(w, h) * 0.30
    dr.ellipse([cx - r * 1.1, cy + r * 0.1, cx + r * 1.1, cy + r * 0.9], fill=(0, 0, 0, 60))
    dr.ellipse([cx - r * 0.85, cy + r * 0.0, cx + r * 0.85, cy + r * 1.0], fill=(107, 79, 140), outline=(14, 18, 22))
    dr.ellipse([cx - r * 0.55, cy - r * 0.8, cx + r * 0.55, cy + r * 0.3], fill=(202, 162, 122), outline=(14, 18, 22))
    return _poly_from_ellipse(cx, cy, w * 0.42, h * 0.42, 14)


def _draw_tree(dr, box):
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    r = min(w, h) * 0.5
    dr.ellipse([cx - r * 0.95, cy - r * 0.9, cx + r * 0.95, cy + r * 0.95], fill=(79, 122, 67), outline=(47, 74, 40))
    dr.ellipse([cx - r * 0.6, cy - r * 0.55, cx + r * 0.2, cy + r * 0.1], fill=(95, 143, 80))
    return _poly_from_ellipse(cx, cy, r * 0.9, r * 0.9, 16)


def _backdrop(img, dr, kind, rng):
    W, H = img.size
    if kind == "field":
        dr.rectangle([0, 0, W, H], fill=FIELD)
        for yy in range(8, H, 15):
            dr.line([(0, yy), (W, yy)], fill=CROP, width=6)
        # a curved dirt track
        pts = [(0, H * 0.72)]
        for t in range(1, 11):
            pts.append((W * t / 10, H * (0.62 + 0.12 * math.sin(t))))
        dr.line(pts, fill=DIRT, width=18)
    else:
        dr.rectangle([0, 0, W, H], fill=ASPHALT)
        for x in range(40, W - 20, 52):
            dr.line([(x, 14), (x, H * 0.42)], fill=LANE, width=2)
            dr.line([(x, H * 0.58), (x, H - 14)], fill=LANE, width=2)
        for x in range(0, W, 26):  # dashed centre caution line
            dr.line([(x, H * 0.5), (x + 14, H * 0.5)], fill=CAUTION, width=2)


def _overlaps(box, placed, slack=-6):
    x, y, w, h = box
    for (px, py, pw, ph) in placed:
        if (x < px + pw - slack and x + w > px + slack and
                y < py + ph - slack and y + h > py + slack):
            return True
    return False


def make_scene(seed=0, kind=None, size=(W_DEF, H_DEF)):
    """Return (PIL.Image RGB, annotations). Each annotation:
       {category_id, bbox:[x,y,w,h], segmentation:[[...]], area, iscrowd}."""
    rng = random.Random(seed)
    if kind is None:
        kind = rng.choice(["lot", "field"])
    W, H = size
    img = Image.new("RGB", (W, H))
    dr = ImageDraw.Draw(img, "RGBA")
    _backdrop(img, dr, kind, rng)

    anns, placed = [], []
    n_cars = rng.randint(3, 6)
    n_people = rng.randint(2, 5)
    n_trees = rng.randint(1, 3)

    def place(cat, wr, hr, tries=40):
        for _ in range(tries):
            w = rng.uniform(*wr); h = rng.uniform(*hr)
            if rng.random() < 0.5 and cat == 1:  # some cars rotated to horizontal
                w, h = h, w
            x = rng.uniform(6, W - w - 6); y = rng.uniform(6, H - h - 6)
            box = (x, y, w, h)
            if not _overlaps(box, placed):
                placed.append(box); return box
        return None

    for _ in range(n_cars):
        b = place(1, (60, 96), (86, 120))
        if b:
            poly = _draw_car(dr, b, rng.choice(CAR_COLORS))
            anns.append(_ann(1, b, poly))
    for _ in range(n_people):  # small from altitude
        b = place(2, (14, 26), (18, 34))
        if b:
            poly = _draw_person(dr, b)
            anns.append(_ann(2, b, poly))
    for _ in range(n_trees):
        b = place(3, (54, 90), (54, 90))
        if b:
            poly = _draw_tree(dr, b)
            anns.append(_ann(3, b, poly))
    return img, anns


def _ann(cat, box, poly):
    x, y, w, h = box
    seg = [round(float(v), 1) for pt in poly for v in pt]
    return {"category_id": cat, "bbox": [round(float(x), 1), round(float(y), 1),
            round(float(w), 1), round(float(h), 1)],
            "segmentation": [seg], "area": float(w * h), "iscrowd": 0}


def build_coco(n=40, out_dir="drone_coco", split="train", size=(W_DEF, H_DEF), seed0=0):
    """Write n images + one COCO json. Returns (json_path, img_dir)."""
    img_dir = os.path.join(out_dir, split)
    os.makedirs(img_dir, exist_ok=True)
    images, annotations = [], []
    ann_id = 1
    for i in range(n):
        img, anns = make_scene(seed=seed0 + i, size=size)
        fn = f"{split}_{i:04d}.png"
        img.save(os.path.join(img_dir, fn))
        images.append({"id": i, "file_name": fn, "width": size[0], "height": size[1]})
        for a in anns:
            a = dict(a); a["id"] = ann_id; a["image_id"] = i
            annotations.append(a); ann_id += 1
    coco = {"images": images, "annotations": annotations,
            "categories": [{"id": i + 1, "name": c} for i, c in enumerate(THING_CLASSES)]}
    jp = os.path.join(out_dir, f"{split}.json")
    with open(jp, "w") as f:
        json.dump(coco, f)
    return jp, img_dir


# convenience: numpy image + boxes for the metric notebooks
def scene_arrays(seed=0, kind="lot", size=(W_DEF, H_DEF)):
    img, anns = make_scene(seed, kind, size)
    boxes = np.array([a["bbox"] for a in anns], dtype=float)  # xywh
    labels = np.array([a["category_id"] for a in anns], dtype=int)
    return np.asarray(img), boxes, labels


In [ ]:
%%writefile det_metrics.py
"""det_metrics.py — from-scratch detection primitives, in the same
conventions Detectron2 uses, so the notebooks can show the hand math
next to the library call. Boxes are xywh (COCO) unless noted."""
import numpy as np


def xywh_to_xyxy(b):
    b = np.asarray(b, float)
    return np.stack([b[..., 0], b[..., 1], b[..., 0] + b[..., 2], b[..., 1] + b[..., 3]], -1)


def iou_matrix(a_xywh, b_xywh):
    """Pairwise IoU between two sets. Returns [len(a), len(b)]."""
    a = xywh_to_xyxy(np.atleast_2d(a_xywh)); b = xywh_to_xyxy(np.atleast_2d(b_xywh))
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def iou(a, b):
    return float(iou_matrix([a], [b])[0, 0])


def nms(boxes_xywh, scores, iou_thresh=0.5):
    """Greedy non-maximum suppression. Returns kept indices (by score)."""
    idx = np.argsort(scores)[::-1]
    keep = []
    while len(idx):
        i = idx[0]; keep.append(i)
        if len(idx) == 1:
            break
        ious = iou_matrix([boxes_xywh[i]], boxes_xywh[idx[1:]])[0]
        idx = idx[1:][ious < iou_thresh]
    return keep


def average_precision(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    """Single-class AP (area under P-R). Greedy TP/FP assignment by
    descending score, one GT per prediction. Returns (ap, recall, precision)."""
    order = np.argsort(pred_scores)[::-1]
    n_gt = len(gt_boxes)
    matched = np.zeros(n_gt, bool)
    tp = np.zeros(len(order)); fp = np.zeros(len(order))
    for rank, pi in enumerate(order):
        if n_gt == 0:
            fp[rank] = 1; continue
        ious = iou_matrix([pred_boxes[pi]], gt_boxes)[0]
        j = int(np.argmax(ious))
        if ious[j] >= iou_thresh and not matched[j]:
            tp[rank] = 1; matched[j] = True
        else:
            fp[rank] = 1
    tp_c = np.cumsum(tp); fp_c = np.cumsum(fp)
    recall = tp_c / max(n_gt, 1)
    precision = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    # AP = area under P-R with monotone-decreasing precision envelope
    mrec = np.concatenate([[0], recall, [recall[-1] if len(recall) else 0]])
    mpre = np.concatenate([[1], precision, [0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    return ap, recall, precision


def encode_deltas(anchor_xywh, gt_xywh):
    """RPN/Faster R-CNN box regression targets (tx,ty,tw,th)."""
    ax, ay, aw, ah = anchor_xywh
    acx, acy = ax + aw / 2, ay + ah / 2
    gx, gy, gw, gh = gt_xywh
    gcx, gcy = gx + gw / 2, gy + gh / 2
    return (( gcx - acx) / aw, (gcy - acy) / ah, np.log(gw / aw), np.log(gh / ah))


In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mp
from drone_synth import scene_arrays
from det_metrics import iou_matrix, encode_deltas

### Place 9 anchors (3 scales × 3 ratios) at one feature location

In [ ]:
img, boxes, labels = scene_arrays(seed=3, kind='lot')
target = boxes[np.argmax(boxes[:,2]*boxes[:,3])]     # a car
cx, cy = target[0]+target[2]/2, target[1]+target[3]/2  # put location on the car
scales = [40, 72, 116]; ratios = [1.0, 1.6, 0.625]     # 1:1, 1.6:1, 1:1.6
anchors = []
for s in scales:
    for r in ratios:
        w = s*np.sqrt(r); h = s/np.sqrt(r)
        anchors.append([cx-w/2, cy-h/2, w, h])
anchors = np.array(anchors)
ious = iou_matrix(anchors, [target])[:,0]
best = int(np.argmax(ious))
print('best anchor idx', best, 'IoU', round(ious[best],3))

### Which prior fits the car best?

In [ ]:
fig, ax = plt.subplots(figsize=(6.5,6.5)); ax.imshow(img); ax.axis('off')
ax.add_patch(mp.Rectangle(target[:2], target[2], target[3], fill=False, edgecolor='#1E5F8C', lw=2.5))
for i,a in enumerate(anchors):
    c = '#B3157E' if i==best else '#4C5548'
    lw = 2.4 if i==best else 1
    ax.add_patch(mp.Rectangle(a[:2], a[2], a[3], fill=False, edgecolor=c, lw=lw, ls='--', alpha=1 if i==best else .5))
ax.plot(cx, cy, 'o', color='#B3157E')
ax.set_title(f'blue=GT car · magenta=best anchor (IoU {ious[best]:.2f})'); plt.show()

### The regression target the network must learn

In [ ]:
tx,ty,tw,th = encode_deltas(anchors[best], target)
print('best anchor xywh:', anchors[best].round(1))
print('target      xywh:', target.round(1))
print(f'deltas  tx={tx:.3f}  ty={ty:.3f}  tw={tw:.3f}  th={th:.3f}')
print('The head predicts these 4 small numbers, not raw pixel coords.')

### Detectron2 equivalent (optional): `DefaultAnchorGenerator`
Detectron2 tiles anchors across a whole feature map. Here we generate them for one pyramid level and overlay the grid on the drone scene.

In [ ]:
# --- OPTIONAL: install Detectron2 to run the "library equivalent" cells ---
# This notebook's teaching content runs fully on numpy/Pillow WITHOUT this.
# Run this only if you also want the Detectron2 API demonstrations.
!python -m pip install -q 'pyyaml==6.0.*'
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
try:
    import detectron2; print("detectron2", detectron2.__version__, "ready")
except Exception as e:
    print("Detectron2 not available; the numpy cells still work.\n", e)


In [ ]:
try:
    import torch
    from detectron2.modeling.anchor_generator import DefaultAnchorGenerator
    from detectron2.config import get_cfg
    # a feature map stride of 16 over the 512x512 image -> 32x32 grid
    class F:  # minimal shape spec
        def __init__(s,c,st): s.channels=c; s.stride=st
    gen = DefaultAnchorGenerator(sizes=[[72]], aspect_ratios=[[0.625,1.0,1.6]], strides=[32])
    feat = torch.zeros(1,1,16,16)   # 16x16 locations at stride 32 -> 512px
    cells = gen([feat])[0].tensor.numpy()
    print('generated', len(cells), 'anchors (16x16 grid x 3 ratios)')
    fig, ax = plt.subplots(figsize=(6,6)); ax.imshow(img); ax.axis('off')
    for a in cells[::7]:
        ax.add_patch(mp.Rectangle((a[0],a[1]), a[2]-a[0], a[3]-a[1], fill=False, edgecolor='#B3157E', lw=.5, alpha=.5))
    ax.set_title('DefaultAnchorGenerator tiling (subsampled)'); plt.show()
except Exception as e:
    print('skipped:', e)